# Day 11 — Math & Collection Functions

**What you will learn:**
- Math functions: `round`, `ceil`, `floor`, `abs`, `pow`, `sqrt`, `log`
- Array operations: `array`, `explode`, `collect_list`, `array_contains`, `array_distinct`
- Map operations: `map`, `map_keys`, `map_values`, `element_at`
- Struct operations: `struct`, field access

**Exam relevance:** DataFrame API (30%) — `explode`, `collect_list`, `array_contains` are exam favourites.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## 1. Math Functions

In [ ]:
df = spark.createDataFrame([
    (1, 3.7,  -42.5, 9.0),
    (2, 1.2,   17.8, 16.0),
    (3, 8.99, -3.1,  25.0),
], ["id", "score", "delta", "area"])

df.select(
    col("score"),
    round(col("score"), 1).alias("rounded"),
    ceil(col("score")).alias("ceil"),
    floor(col("score")).alias("floor"),
    abs(col("delta")).alias("abs_delta"),
    sqrt(col("area")).alias("sqrt_area"),
    pow(col("score"), 2).alias("score_sq"),
    log(col("area")).alias("ln_area"),
).show()

## 2. Arrays — Creating and Querying

In [ ]:
df_arr = spark.createDataFrame([
    (1, "Alice",   ["python", "sql", "spark"]),
    (2, "Bob",     ["java", "scala", "sql"]),
    (3, "Carol",   ["python", "r", "sql"]),
    (4, "Dave",    ["spark", "python"]),
    (5, "Eve",     []),
], ["id", "name", "skills"])
df_arr.show(truncate=False)

# Array size and element access
df_arr.select(
    col("name"),
    size(col("skills")).alias("skill_count"),
    col("skills")[0].alias("first_skill"),
    array_contains(col("skills"), "python").alias("knows_python"),
).show()

## 3. explode — Array Rows to Individual Rows

> **Exam tip:** `explode` drops rows where the array is null or empty. `explode_outer` keeps them as a null row.

In [ ]:
# explode — one row per skill
print("=== explode ===")
df_arr.select(col("name"), explode(col("skills")).alias("skill")).show()

# explode_outer — keeps Eve's empty row as null
print("=== explode_outer ===")
df_arr.select(col("name"), explode_outer(col("skills")).alias("skill")).show()

# posexplode — also returns the index
print("=== posexplode ===")
df_arr.select(col("name"), posexplode(col("skills")).alias("pos", "skill")).show()

## 4. collect_list & collect_set — Rows to Array

The inverse of `explode` — aggregate multiple rows into an array.

In [ ]:
flat = df_arr.select(col("name"), explode(col("skills")).alias("skill"))

# collect_list — preserves duplicates
flat.groupBy("skill").agg(collect_list("name").alias("people_with_skill")).orderBy("skill").show(truncate=False)

# collect_set — deduplicates
flat.groupBy("name").agg(collect_set("skill").alias("unique_skills")).show(truncate=False)

## 5. Array Manipulation Functions

In [ ]:
df_arr.select(
    col("name"),
    array_distinct(col("skills")).alias("distinct_skills"),
    array_sort(col("skills")).alias("sorted_skills"),
    array_union(col("skills"), array(lit("databricks"))).alias("with_databricks"),
).show(truncate=False)

## 6. Maps — Key-Value Pairs

In [ ]:
df_map = spark.createDataFrame([
    (1, "Alice", {"python": 5, "sql": 4, "spark": 3}),
    (2, "Bob",   {"java": 4,   "scala": 5}),
    (3, "Carol", {"python": 3, "r": 4}),
], ["id", "name", "skill_scores"])
df_map.printSchema()

df_map.select(
    col("name"),
    map_keys(col("skill_scores")).alias("skills"),
    map_values(col("skill_scores")).alias("scores"),
    element_at(col("skill_scores"), "python").alias("python_score"),
    col("skill_scores")["java"].alias("java_score"),
    size(col("skill_scores")).alias("num_skills"),
).show(truncate=False)

## 7. Structs — Nested Row Objects

In [ ]:
from pyspark.sql.functions import struct

df = spark.createDataFrame([
    ("Alice", 34, "RO", 95000),
    ("Bob",   28, "UK", 72000),
], ["name", "age", "country", "salary"])

# Pack columns into a struct
df_struct = df.withColumn("location", struct(col("country"), col("age")))
df_struct.printSchema()
df_struct.show()

# Access struct fields using dot notation
df_struct.select(col("name"), col("location.country"), col("location.age")).show()

---

## Day 11 Checklist

- [ ] Used `round`, `ceil`, `floor`, `abs`, `sqrt`
- [ ] Used `array_contains`, `size`, indexed element access on arrays
- [ ] Used `explode` vs `explode_outer` — know the difference
- [ ] Used `posexplode` to get index + value
- [ ] Used `collect_list` and `collect_set` to aggregate rows into arrays
- [ ] Used `map_keys`, `map_values`, `element_at` on a map column
- [ ] Created a struct with `struct()` and accessed fields with dot notation

**Next:** Day 12 — UDFs (User Defined Functions)